# Selecting a location for a well

Let's say we work for the mining company GlavRosGosNeft. We need to decide where to drill a new well.

We were provided with oil samples in three regions: in each 10,000 fields, where the quality of oil and the volume of its reserves were measured. We will build a machine learning model that will help determine the region where mining will bring the greatest profit. Let's analyze the possible profits and risks using the *Bootstrap.* technique

Steps to select a location:

- Deposits are searched for in the selected region, and the characteristic values are determined for each;
- Build a model and estimate the volume of reserves;
- Deposits with the highest estimated values ​​are selected. The number of fields depends on the company’s budget and the cost of developing one well;
- Profit is equal to the total profit of the selected fields.

# Selecting a location for a well

* The purpose of this study is to figure out where it will be most profitable to drill a new well. We will need this to more clearly allocate the budget to locations with high income from them. Datasets with regional wells were provided for analysis.
* Action plan:
    * Get dataset
    * Train the model
    * Assess the quality of models and perform calculations
    * Use Bootstrap technique

## Loading and preparing data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import  LinearRegression
from sklearn.metrics import f1_score
from sklearn.utils import shuffle
from sklearn.metrics import roc_curve
from sklearn.metrics import mean_squared_error
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
%matplotlib inline

In [4]:
first_reg = pd.read_csv('../datasets/geo_data_0.csv')
second_reg = pd.read_csv('../datasets/geo_data_1.csv')
third_reg = pd.read_csv('../datasets/geo_data_2.csv')

In [5]:
def information(data):
    data.info()
    display(data.head(20))
    display(data.describe())

In [6]:
information(first_reg)
information(second_reg)
information(third_reg)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647
5,wX4Hy,0.969570,0.489775,-0.735383,64.741541
6,tL6pL,0.645075,0.530656,1.780266,49.055285
7,BYPU6,-0.400648,0.808337,-5.624670,72.943292
8,j9Oui,0.643105,-0.551583,2.372141,113.356160
9,OLuZU,2.173381,0.563698,9.441852,127.910945


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.500419,0.250143,2.502647,92.500000
std,0.871832,0.504433,3.248248,44.288691
min,-1.408605,-0.848218,-12.088328,0.000000
25%,-0.072580,-0.200881,0.287748,56.497507
50%,0.502360,0.250252,2.515969,91.849972
75%,1.073581,0.700646,4.715088,128.564089
max,2.362331,1.343769,16.003790,185.364347


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305
5,HHckp,-3.327590,-2.205276,3.003647,84.038886
6,h5Ujo,-11.142655,-10.133399,4.002382,110.992147
7,muH9x,4.234715,-0.001354,2.004588,53.906522
8,YiRkx,13.355129,-0.332068,4.998647,134.766305
9,jG6Gi,1.069227,-11.025667,4.997844,137.945408


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,1.141296,-4.796579,2.494541,68.825000
std,8.965932,5.119872,1.703572,45.944423
min,-31.609576,-26.358598,-0.018144,0.000000
25%,-6.298551,-8.267985,1.000021,26.953261
50%,1.153055,-4.813172,2.011479,57.085625
75%,8.621015,-1.332816,3.999904,107.813044
max,29.421755,18.734063,5.019721,137.945408


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746
5,LzZXx,-0.758092,0.710691,2.585887,90.222465
6,WBHRv,-0.574891,0.317727,1.773745,45.641478
7,XO8fn,-1.906649,-2.458350,-0.177097,72.480640
8,ybmQ5,1.776292,-0.279356,3.004156,106.616832
9,OilcN,-1.214452,-0.439314,5.922514,52.954532


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.002023,-0.002081,2.495128,95.000000
std,1.732045,1.730417,3.473445,44.749921
min,-8.760004,-7.084020,-11.970335,0.000000
25%,-1.162288,-1.174820,0.130359,59.450441
50%,0.009424,-0.009482,2.484236,94.925613
75%,1.158535,1.163678,4.858794,130.595027
max,7.238262,7.844801,16.739402,190.029838


In [7]:
# data = pd.concat([first_reg] + [second_reg] + [third_reg])
# data

Based on a preliminary inspection of the datasets, they are in order. There are no gaps in the data, so you don’t have to fill in anything.

## Model training and testing

First of all, let's split the datasets into target and non-target features.

In [8]:
first_reg_features = first_reg.drop('product', axis=1)
second_reg_features = second_reg.drop('product', axis=1)
third_reg_features = third_reg.drop('product', axis=1)

first_reg_target = first_reg['product']
second_reg_target = second_reg['product']
third_reg_target = third_reg['product']

Let's write a function in which we train models using linear regression, because it is quite predictable, and also calculate the root mean square error of our models.

In [9]:
def ml_and_checking(features, target, region):
    features = features.drop('id', axis=1)
    scaler = StandardScaler()
    numerics = ['f0', 'f1', 'f2']
    scaler.fit_transform(features[numerics])
    features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=0.25, random_state=12345)
    model = LinearRegression()
    model.fit(features_train, target_train)
    predicted = model.predict(features_valid)
    print(f'Average predicted reserve volume for {region}:', predicted.mean())
    print(f'RMSE for {region}:', mean_squared_error(y_true=target_valid, y_pred=predicted, squared=False))
    print()
    return pd.concat([pd.Series(predicted)] + [pd.Series(model.predict(features_train))]), pd.concat([pd.Series(target_valid)] + [pd.Series(target_train)])

In [10]:
data = pd.DataFrame([], columns=['original', 'predicted'])
predicted_first_reg, original_first_reg = ml_and_checking(first_reg_features, first_reg_target, 'first_region')
predicted_second_reg, original_second_reg = ml_and_checking(second_reg_features, second_reg_target, 'second_region')
predicted_third_reg, original_third_reg = ml_and_checking(third_reg_features, third_reg_target, 'third_region')
predicted_data = pd.concat([predicted_first_reg] + [predicted_second_reg] + [predicted_third_reg])
original_data = pd.concat([original_first_reg] + [original_second_reg] + [original_third_reg])
data['original'] = list(original_data)
data['predicted'] = list(predicted_data)


Average predicted reserve volume first_region: 92.59256778438035
Model RMSE first_region: 37.5794217150813

Average predicted reserve volume second_region: 68.728546895446
Model RMSE second_region: 0.893099286775617

Average predicted reserve volume third_region: 94.96504596800489
Model RMSE third_region: 40.02970873393434



You can see that the model performed best in the second region, *RMSE* is only 0.89! However, the average stock lies in the third region - 94.96.

## Preparation for profit calculation

Let's put into variables the values ​​we have of the total budget, the number of wells and the income per unit of product.

In [11]:
TOTAL_BUDGET = 10000000000
EXPLORE_REGION = 200
SELECTING_BOREHOLES = 500
INCOME_FROM_EACH_UNIT = 450000
NEEDED_VOLUME = TOTAL_BUDGET / EXPLORE_REGION / INCOME_FROM_EACH_UNIT
NEEDED_VOLUME

111.11111111111111

The required number of units of product for break-even is 111.111, however, the average inventory of predicted raw materials is less than 111.111, this may indicate high risks in further calculation of profit.

We will also create a function for calculating profits from wells.

In [12]:
def revenue(best_boreholes):
    bore = best_boreholes
    tmp = data.query('predicted in list(@bore)')
    return tmp['original'].sum() * INCOME_FROM_EACH_UNIT - TOTAL_BUDGET


## Calculation of profits and risks

Let's create a function for selecting the two hundred best wells.

In [13]:
def best_boreholes(predicted_boreholes):
    top_200_boreholes = predicted_boreholes.sort_values(ascending=False, ignore_index=True).head(200)
    return top_200_boreholes


And now the most important thing is that we will implement the *Bootstrap* technique, where we will create samples of 500 wells 1000 times, select the best 200 from them and calculate the 95% confidence interval for profit

In [14]:
def bootstrap(predicted_boreholes, region):
    state = np.random.RandomState(12345)
    values = []
    for i in range(2000):
        subsample = predicted_boreholes.sample(n=SELECTING_BOREHOLES, replace=True, random_state=state)
        subsample = best_boreholes(subsample)
        profit = revenue(subsample)
        values.append(profit)
    values = pd.Series(values)
    mean_profit = values.mean()
    print(f'FOR REGION {region.upper()}')
    print(' Average profit:', mean_profit / 1000000, 'million')
    lower = values.quantile(0.025)
    upper = values.quantile(0.975)
    print(' Lower bound:', lower / 1000000, 'million')
    print(' Upper bound:', upper / 1000000, 'million')
    print(f'  Risk percentage: {(len(values[values < 0]) / len(values)):.2%}')




In [15]:
bootstrap(predicted_first_reg, 'first')
bootstrap(predicted_second_reg, 'second')
bootstrap(predicted_third_reg, 'third')


FOR REGION FIRST
 Average profit: 406.5553449376348 million
 Lower bound: -100.84712478378191 million
 Upper bound: 902.9054765737624 million
  Risk percentage: 6.05%
FOR REGION SECOND
 Average profit: 427.60252158354564 million
 Lower bound: 7.500650001686765 million
 Upper bound: 857.2312306097795 million
  Risk percentage: 2.25%
FOR REGION THIRD
 Average profit: 351.0690438256394 million
 Lower bound: -175.3711348021004 million
 Upper bound: 870.127178241549 million
  Risk percentage: 9.85%


The highest average profit in the second region is 427 million, and the risk there is the lowest - 2.25%!

However, the highest average profit in the second region is 427 million and the risk is 2.25%, which is less than 2.5% according to the condition.

# Conclusion

In this project, work was carried out to obtain data, obtain information about them and further use them in training linear regression models. The models were trained successfully, especially on the data from the second region, the RMSE there is only 0.89! The required volume of wells to make a profit was calculated, and a profit calculation function was created. We implemented the *Boostrap* technique, in which we created samples of 500 wells 1000 times and calculated the necessary metrics. Thus, our eye fell on the second region, because it is there that the highest average profit is with a risk below 2.5%!